# Stream 2 — CDA training (recovered from original Colab notebook)

These cells are extracted from the original notebook. They are **training code, not saved model weights**. Run only the experiment needed, with a T4 GPU and the project ZIP uploaded. Each full training run uses `max_steps=500` (not a fixed epoch count). The original notebook also contains separate evaluation/control code.


In [ ]:
from pathlib import Path
import zipfile, os
zip_path = Path('/content/nlp-gender-bias-urdu-main.zip')
if not Path('/content/nlp-gender-bias-urdu-main').exists():
    assert zip_path.exists(), 'Upload nlp-gender-bias-urdu-main.zip to /content first'
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')
os.chdir('/content/nlp-gender-bias-urdu-main')
print('Project:', os.getcwd())


## CDA-English — 9,972 original/counterfactual pairs (19,944 sentences)


In [ ]:

import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

MODEL = "xlm-roberta-base"
OUTPUT = "/content/cda_en_full"

# Load all English CDA pairs
sentences = []

with open("data/cda/en_pairs.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        sentences.extend([row["en"], row["en_cf"]])

print("Training sentences:", len(sentences))

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForMaskedLM.from_pretrained(MODEL)

dataset = Dataset.from_dict({"text": sentences})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        return_special_tokens_mask=True,
    )

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15,
)

args = TrainingArguments(
    output_dir=OUTPUT,
    max_steps=500,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    fp16=True,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
    processing_class=tokenizer,
)

trainer.train()

model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)

print("FULL CDA-EN TRAINING COMPLETED!")
print("Saved model:", OUTPUT)


## CDA-Urdu — filtered Urdu counterfactual pairs


In [ ]:

import os, sys, json, gc, subprocess, re, random
from pathlib import Path
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)

# ---------- CONFIG ----------
os.chdir("/content/nlp-gender-bias-urdu-main")

BASE_MODEL = "xlm-roberta-base"
OUTPUT = "/content/cda_ur_full"
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Enable T4 GPU in Colab!"
assert Path("data/cda/pairs_ur.jsonl").exists()

# ---------- LOAD AND FILTER URDU PAIRS ----------
rows = []
with open("data/cda/pairs_ur.jsonl", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print("Original pairs:", len(rows))

def mostly_latin(text):
    latin = len(re.findall(r"[A-Za-z]", text))
    urdu = len(re.findall(r"[\u0600-\u06FF]", text))
    return latin > urdu

failed = []
identical = []
valid = []

for i, row in enumerate(rows):
    ur = row["ur"].strip()
    cf = row["ur_cf"].strip()

    if not ur or not cf or mostly_latin(ur) or mostly_latin(cf):
        failed.append(row.get("id", i))
    elif ur == cf:
        identical.append(row.get("id", i))
    else:
        valid.append(row)

print("Mostly-Latin/empty pairs removed:", len(failed))
print("Excluded IDs:", failed)
print("Identical pairs removed:", len(identical))
print("Valid CDA pairs:", len(valid))

# Keep original and counterfactual together
random.shuffle(valid)

sentences = []
for row in valid:
    sentences.extend([row["ur"].strip(), row["ur_cf"].strip()])

print("Training sentences:", len(sentences))
assert sentences, "No valid training sentences!"

# ---------- TOKENIZATION ----------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)

dataset = Dataset.from_dict({"text": sentences})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        return_special_tokens_mask=True
    )

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# ---------- TRAINING ----------
args = TrainingArguments(
    output_dir=OUTPUT,
    max_steps=500,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    fp16=True,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
    processing_class=tokenizer
)

print("\nSTARTING URDU CDA TRAINING...")
trainer.train()

model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)

# Save experiment metadata
metadata = {
    "base_model": BASE_MODEL,
    "method": "CDA-UR",
    "seed": SEED,
    "total_pairs": len(rows),
    "failed_pairs_removed": len(failed),
    "failed_pair_ids": failed,
    "identical_pairs_removed": len(identical),
    "training_pairs": len(valid),
    "training_sentences": len(sentences),
    "max_steps": 500,
    "learning_rate": 2e-5,
    "batch_size": 4,
    "gradient_accumulation_steps": 2,
    "torch": torch.__version__
}

Path(OUTPUT, "experiment.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("\nURDU CDA TRAINING COMPLETE!")

# ---------- RELEASE GPU MEMORY ----------
del trainer, model
gc.collect()
torch.cuda.empty_cache()

# ---------- DISCO EVALUATION ----------
experiments = [
    ("en", None, "cda_ur"),
    ("ur", "data/templates/v2", "cda_ur_v2")
]

for lang, templates, tag in experiments:
    cmd = [
        sys.executable, "-m", "src.disco.run",
        "--lang", lang,
        "--model", OUTPUT,
        "--tokenizer", BASE_MODEL,
        "--tag", tag
    ]

    if templates:
        cmd += ["--templates_dir", templates]

    print(f"\nEvaluating CDA-UR on {lang}...", flush=True)

    subprocess.run(cmd, check=True)

# ---------- PRINT RESULTS ----------
print("\n========== FINAL RESULTS ==========")

baselines = {"en": 0.773, "ur": 0.877}

for lang, templates, tag in experiments:
    expected = (
        f"disco_{lang}_cda_ur_full_{tag}.json"
    )
    path = Path("results") / expected

    if not path.exists():
        print("Result not found:", path)
        continue

    result = json.loads(path.read_text(encoding="utf-8"))
    score = result["score"]
    baseline = baselines[lang]

    print(f"\nLanguage: {lang}")
    print(f"Baseline: {baseline:.4f}")
    print(f"After CDA-UR: {score:.4f}")
    print(f"Absolute change: {score-baseline:+.4f}")
    print(f"Relative change: {(score-baseline)/baseline*100:+.2f}%")

print("\nALL DONE!")
print("Model:", OUTPUT)


## CDA-Combined — balanced English and filtered Urdu pairs


In [ ]:

import os, sys, json, re, random, gc, subprocess
from pathlib import Path
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)

os.chdir("/content/nlp-gender-bias-urdu-main")

BASE = "xlm-roberta-base"
OUTPUT = "/content/cda_en_ur_full"
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Enable T4 GPU first!"

# Load English pairs
with open("data/cda/en_pairs.jsonl", encoding="utf-8") as f:
    en_rows = [json.loads(x) for x in f if x.strip()]

# Load Urdu pairs
with open("data/cda/pairs_ur.jsonl", encoding="utf-8") as f:
    ur_rows = [json.loads(x) for x in f if x.strip()]

def mostly_latin(text):
    latin = len(re.findall(r"[A-Za-z]", text))
    arabic = len(re.findall(r"[\u0600-\u06FF]", text))
    return latin > arabic

# Filter Urdu translations
ur_valid = [
    r for r in ur_rows
    if r["ur"].strip()
    and r["ur_cf"].strip()
    and not mostly_latin(r["ur"])
    and not mostly_latin(r["ur_cf"])
    and r["ur"].strip() != r["ur_cf"].strip()
]

# Balance languages by number of pairs
n = min(len(en_rows), len(ur_valid))
rng = random.Random(SEED)
en_sample = rng.sample(en_rows, n)
ur_sample = rng.sample(ur_valid, n)

sentences = []

for r in en_sample:
    sentences.extend([r["en"], r["en_cf"]])

for r in ur_sample:
    sentences.extend([r["ur"], r["ur_cf"]])

rng.shuffle(sentences)

print("English pairs:", n)
print("Urdu pairs:", n)
print("Total sentences:", len(sentences))

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForMaskedLM.from_pretrained(BASE)

dataset = Dataset.from_dict({"text": sentences})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        return_special_tokens_mask=True
    )

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Train
args = TrainingArguments(
    output_dir=OUTPUT,
    max_steps=500,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    fp16=True,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
    processing_class=tokenizer
)

print("\nSTARTING COMBINED CDA TRAINING...")
trainer.train()

model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)

Path(OUTPUT, "experiment.json").write_text(
    json.dumps({
        "method": "CDA-EN+UR",
        "base_model": BASE,
        "seed": SEED,
        "english_pairs": n,
        "urdu_pairs": n,
        "total_sentences": len(sentences),
        "max_steps": 500,
        "learning_rate": 2e-5,
        "urdu_identical_pairs_excluded": True
    }, indent=2),
    encoding="utf-8"
)

print("\nCOMBINED CDA TRAINING COMPLETE!")

del trainer, model
gc.collect()
torch.cuda.empty_cache()

# Evaluate both languages
for lang, templates, tag in [
    ("en", None, "cda_en_ur"),
    ("ur", "data/templates/v2", "cda_en_ur_v2")
]:
    cmd = [
        sys.executable, "-m", "src.disco.run",
        "--lang", lang,
        "--model", OUTPUT,
        "--tokenizer", BASE,
        "--tag", tag
    ]

    if templates:
        cmd += ["--templates_dir", templates]

    print(f"\nEvaluating {lang}...", flush=True)
    subprocess.run(cmd, check=True)

# Print scores
print("\n===== COMBINED CDA RESULTS =====")

for lang, tag, baseline in [
    ("en", "cda_en_ur", 0.773),
    ("ur", "cda_en_ur_v2", 0.877)
]:
    path = Path(
        f"results/disco_{lang}_cda_en_ur_full_{tag}.json"
    )

    if path.exists():
        result = json.loads(path.read_text(encoding="utf-8"))
        score = result["score"]
        print(
            f"{lang.upper()}: baseline={baseline:.4f}, "
            f"combined={score:.4f}, "
            f"change={score-baseline:+.4f}"
        )
    else:
        print("Result file not found:", path)

print("\nDONE!")
print("Saved model:", OUTPUT)


## Reproducibility notes

- Base model: `xlm-roberta-base`; masked-language modeling with masking probability 0.15.
- Learning rate: 2e-5; per-device batch size: 4; gradient accumulation: 2 (effective batch 8 on one GPU); maximum sequence length: 128; 500 optimizer steps; fp16; seed 42.
- Hardware used in the original experiment: Tesla T4 GPU.
- Epoch count is not specified; training is controlled by `max_steps=500`.
- The English experiment uses 19,944 sentences before tokenization; Urdu and combined counts are printed by their cells after filtering.
- Model directories under `/content` are ephemeral. Back up resulting model files externally.
- The Urdu and combined cells also launch DisCo evaluations after training.
